# H1: the conditional scale is tracked, the tail shape is not

Every table the findings quote for H1, recomputed here from the CSVs in
`results/h1_shape_vs_scale/`. Where a number disagrees with the text, the cell says so.

The two halves of the dissociation are measured by the same procedure on the same data,
so the finding is the difference between them, not either level on its own.

In [1]:
import sys
sys.path.insert(0, "..")
import numpy as np, pandas as pd
from common import paths, metrics, tables, stats, quiet
# Narrow filter, not filterwarnings("ignore"): numeric warnings stay visible,
# because one of them hid a divide-by-zero in the tail-index inversion for months.
quiet.silence()
pd.set_option("display.width", 200)
R = lambda n: pd.read_csv(paths.result(n))
print("repo root:", paths.ROOT)


repo root: D:\faks\master\tabular-foundation-model-tails


## The estimators themselves

Two checks before any result: the tail-index inversion must round-trip, and the Hill
estimator must be unbiased. Two of the three copies of Hill in the old repo used the
smallest included order statistic as the threshold, which biases xi low by about xi/k.

In [2]:
xi = 0.7
raw = float(metrics.gpd_quantile(.99, xi) / metrics.gpd_quantile(.9, xi))
res = float((metrics.gpd_quantile(.99, xi) - metrics.gpd_quantile(.5, xi)) /
            (metrics.gpd_quantile(.9, xi) - metrics.gpd_quantile(.5, xi)))
print("round trip, raw ratio      : %.4f  (true %.2f)" % (metrics.xi_from_ratio(raw), xi))
print("round trip, residual ratio : %.4f  (true %.2f)" % (metrics.xi_from_residual_ratio(res), xi))

def hill_biased(y, k_share=0.10, k_min=30):
    y = np.sort(np.asarray(y)[np.asarray(y) > 0])
    k = max(int(len(y) * k_share), k_min)
    return float(np.mean(np.log(y[-k:])) - np.log(y[-k]))

rows = []
for true_xi in (0.5, 0.9):
    for n in (5000, 300):
        old, new = [], []
        for s in range(200):
            z = (1 - np.random.default_rng(s).random(n)) ** (-true_xi)
            old.append(hill_biased(z)); new.append(metrics.hill(z))
        rows.append(dict(true_xi=true_xi, n=n, k=max(int(n * 0.10), 30),
                         biased=np.mean(old), fixed=np.mean(new),
                         difference=np.mean(new) - np.mean(old)))
print()
print(pd.DataFrame(rows).round(4).to_string(index=False))
print()
print("The bias is xi/k, so it bites only at the k=30 floor. It moves no headline")
print("number: the dataset entry test compares three estimators against a 0.25 range.")

round trip, raw ratio      : 0.7000  (true 0.70)
round trip, residual ratio : 0.7000  (true 0.70)



 true_xi    n   k  biased  fixed  difference
     0.5 5000 500  0.4967 0.4976      0.0009
     0.5  300  30  0.4879 0.5062      0.0182
     0.9 5000 500  0.8940 0.8957      0.0017
     0.9  300  30  0.8783 0.9111      0.0328

The bias is xi/k, so it bites only at the k=30 floor. It moves no headline
number: the dataset entry test compares three estimators against a 0.25 range.


## 1. Shape share, synthetic

The shape experiment splits the test set into terciles of x4, which carries the true tail
shape and has zero weight in the scale. The share is the model slope over the true slope.

In [3]:
for name, label in [("shape_of_x_20.csv", "n_est=4 (library default)"),
                    ("shape_of_x_nest1.csv", "n_est=1"),
                    ("shape_of_x_exaone_20.csv", "EXAONE n_est=4"),
                    ("shape_of_x_exaone_nest1.csv", "EXAONE n_est=1")]:
    d = R(name)
    print("--- %s   (%s, %d rows, %d seeds)" % (label, name, len(d), d.seed.nunique()))
    print(tables.slope_by_model(d).round(3).to_string())
    print()

--- n_est=4 (library default)   (shape_of_x_20.csv, 180 rows, 20 seeds)
           model_slope  true_slope  share
model                                    
GBM              0.225       0.497  0.453
TabICLv2         0.213       0.497  0.429
TabPFN-V3        0.096       0.497  0.194

--- n_est=1   (shape_of_x_nest1.csv, 180 rows, 20 seeds)
           model_slope  true_slope  share
model                                    
GBM              0.225       0.497  0.453
TabICLv2         0.221       0.497  0.444
TabPFN-V3        0.211       0.497  0.424

--- EXAONE n_est=4   (shape_of_x_exaone_20.csv, 60 rows, 20 seeds)
        model_slope  true_slope  share
model                                 
EXAONE        0.197       0.497  0.395

--- EXAONE n_est=1   (shape_of_x_exaone_nest1.csv, 60 rows, 20 seeds)
        model_slope  true_slope  share
model                                 
EXAONE        0.183       0.497  0.368



The ensemble size is not a detail: at the library default TabPFN captures 19% of the
shape gradient, at one member 42%. The apparent gap between the two leading models is an
artefact of that setting, which is why the ensemble size is a column and part of the
resume key.

## 2. Scale share, synthetic

In [4]:
scale = pd.concat([R("scale_of_x.csv"), R("scale_of_x_exaone_20.csv")])
print("%d rows, n_est=%s" % (len(scale), sorted(scale.n_est.unique())))
print(tables.scale_share_by_model(scale).round(3).to_string())

240 rows, n_est=[4]
           share
model           
TabICLv2   0.984
TabPFN-V3  0.953
EXAONE     0.929
GBM        0.874


## 3. The dissociation itself

Scale against shape, per model, from the two tables above.

In [5]:
shape4 = tables.slope_by_model(pd.concat([R("shape_of_x_20.csv"),
                                          R("shape_of_x_exaone_20.csv")]))
shape1 = tables.slope_by_model(pd.concat([R("shape_of_x_nest1.csv"),
                                          R("shape_of_x_exaone_nest1.csv")]))
sc = tables.scale_share_by_model(scale)
tab = pd.DataFrame({"scale": sc.share, "shape n_est=4": shape4.share,
                    "shape n_est=1": shape1.share})
tab["scale minus shape (n_est=1)"] = tab["scale"] - tab["shape n_est=1"]
print(tab.round(3).to_string())
print()
print("The findings quote about 94% for scale and 40 to 50% for shape at one member.")
print("Recomputed: scale median %.0f%%, shape median %.0f%%"
      % (100 * tab["scale"].median(), 100 * tab["shape n_est=1"].median()))

           scale  shape n_est=4  shape n_est=1  scale minus shape (n_est=1)
model                                                                      
EXAONE     0.929          0.395          0.368                        0.561
GBM        0.874          0.453          0.453                        0.421
TabICLv2   0.984          0.429          0.444                        0.540
TabPFN-V3  0.953          0.194          0.424                        0.529

The findings quote about 94% for scale and 40 to 50% for shape at one member.
Recomputed: scale median 94%, shape median 43%


## 4. The attainability control

If the tail shape simply could not be estimated from 2000 rows, the deficit would say
nothing about the models. An extreme-value oracle on the same sample recovers 98%.

In [6]:
o = R("oracle_xi_of_x.csv")
print("%d rows, n grid: %s" % (len(o), sorted(o.n.unique())))
rows = []
for n, g in o.groupby("n"):
    t = (g.true_2 - g.true_0).mean()
    r = {"n": int(n), "true": t}
    for pre, lab in [("oracle_s_", "oracle GPD"), ("oracle_r_", "oracle residual"),
                     ("empirical_", "empirical")]:
        r[lab] = (g[pre + "2"] - g[pre + "0"]).mean() / t
    rows.append(r)
t = pd.DataFrame(rows).set_index("n")
print((t[["oracle GPD", "oracle residual", "empirical"]]).round(3).to_string())
print()
print("The models are measured at n_train=2000, so that row is the like-for-like")
print("control: the oracle recovers %.0f%% of the true gradient there, against the"
      % (100 * t.loc[2000, "oracle residual"]))
print("40 to 50%% the models reach. The deficit is not a limit of the sample.")

120 rows, n grid: [500, 1000, 2000, 5000, 20000, 100000]
        oracle GPD  oracle residual  empirical
n                                             
500            NaN              NaN      0.678
1000         0.991            0.967      0.660
2000         0.982            0.976      0.812
5000         0.967            0.986      0.832
20000        0.882            0.897      0.825
100000       0.967            0.987      0.826

The models are measured at n_train=2000, so that row is the like-for-like
control: the oracle recovers 98% of the true gradient there, against the
40 to 50%% the models reach. The deficit is not a limit of the sample.


## 5. Robustness: tail families, sample size, reseeding

In [7]:
f = R("tail_families.csv")
print("tail families: %d runs over %s" % (len(f), sorted(f.family.unique())))
w = f.dropna(subset=["shape_share", "scale_share"])
wins = int(((w.scale_share - 1).abs() < (w.shape_share - 1).abs()).sum())
print("  scale closer to the truth than shape in %d of %d runs" % (wins, len(w)))
print()
sc_ = R("sample_complexity.csv")
print("sample size (shape share by n_train):")
print(tables.ok_rows(sc_).groupby("n_train").shape_share.mean().round(3).to_string())
print()
print("reseeding: TabPFN had random_state=0 while the others got the seed.")
for a, b, lab in [("shape_of_x_20.csv", "shape_of_x_tabpfn_reseed_20.csv", "n_est=4"),
                  ("shape_of_x_nest1.csv", "shape_of_x_tabpfn_reseed_nest1.csv", "n_est=1")]:
    old, new = tables.slope_by_model(R(a)), tables.slope_by_model(R(b))
    if "TabPFN-V3" in old.index and "TabPFN-V3" in new.index:
        print("  %s: before %.0f%%  after %.0f%%"
              % (lab, 100 * old.loc["TabPFN-V3", "share"], 100 * new.loc["TabPFN-V3", "share"]))

tail families: 60 runs over ['burr', 'frechet', 'gpd', 'studentt']
  scale closer to the truth than shape in 60 of 60 runs

sample size (shape share by n_train):
n_train
500     0.214
1000    0.287
2000    0.266
4000    0.561
8000    0.654

reseeding: TabPFN had random_state=0 while the others got the seed.
  n_est=4: before 19%  after 19%
  n_est=1: before 42%  after 43%


## 6. Real data: the claim holds by rank

On real datasets the shape share cannot be quoted as a percentage, because the reference
gradient is weak and the quotient explodes. The claim is therefore that the scale is
closer to the truth than the shape, run by run.

In [8]:
for name in ["dissociation_real.csv", "dissociation_real_exaone.csv"]:
    d = tables.ok_rows(R(name))
    print(name)
    # Two bases, because the findings quote a different one for each file.
    # "paired" scores both estimators on the rows where BOTH returned a value,
    # so the two are comparable; "own rows" scores each on its own valid rows.
    both = d.dropna(subset=["shape_share", "shape_share_residual", "scale_share"])
    for basis, frame in [("paired", both), ("own rows", d)]:
        for col, lab in [("shape_share", "raw ratio"),
                         ("shape_share_residual", "location invariant")]:
            r = tables.paired_rank_test(frame, a="scale_share", b=col)
            print("   %-8s %-20s %d of %d rows, p = %.2g   |   %d of %d datasets, p = %.3g"
                  % (basis, lab, r["a_closer"], r["n"], r["p"],
                     r["cluster_wins"], r["clusters"], r["cluster_p"]))
    print("   datasets %d, models %s" % (d.dataset.nunique(), sorted(d.model.unique())))
    print("   shape share range %.2f to %.2f  <- why this is a rank claim"
          % (d.shape_share.min(), d.shape_share.max()))
    print()

print("QUOTE THE SECOND P-VALUE. The rows are nested: 9 datasets x 3 models x 3 seeds.")
print("A Wilcoxon over 59 of them counts dependent comparisons as independent ones,")
print("which is what produced p = 3.4e-11. At the level of the free unit, the dataset,")
print("the same claim stands at 9 of 9 and p = 0.002. See common/stats.py.")


dissociation_real.csv


   paired   raw ratio            55 of 59 rows, p = 1.1e-10   |   9 of 9 datasets, p = 0.00195
   paired   location invariant   58 of 59 rows, p = 3.4e-11   |   9 of 9 datasets, p = 0.00195
   own rows raw ratio            55 of 59 rows, p = 1.1e-10   |   9 of 9 datasets, p = 0.00195
   own rows location invariant   65 of 66 rows, p = 2.4e-12   |   9 of 9 datasets, p = 0.00195
   datasets 10, models ['GBM', 'TabICLv2', 'TabPFN-V3']
   shape share range -1.57 to 7.27  <- why this is a rank claim

dissociation_real_exaone.csv
   paired   raw ratio            18 of 19 rows, p = 1.9e-05   |   8 of 8 datasets, p = 0.00391
   paired   location invariant   19 of 19 rows, p = 3.8e-06   |   8 of 8 datasets, p = 0.00391
   own rows raw ratio            18 of 19 rows, p = 1.9e-05   |   8 of 8 datasets, p = 0.00391
   own rows location invariant   22 of 22 rows, p = 4.8e-07   |   9 of 9 datasets, p = 0.00195
   datasets 10, models ['EXAONE']
   shape share range -0.48 to 3.40  <- why this is a ran

## 6b. How many free units each p-value rests on

The measurements are nested, and a paired test over all rows treats dependent
comparisons as independent ones. Both levels are printed side by side: the row
level is what the findings quoted, the cluster level is what they should quote.

On the synthetic data the free unit is the seed, because one seed is one draw of
the data and the three models share it. On the real data it is the dataset.


In [9]:
syn = tables.share_by_run(
    pd.concat([R("shape_of_x_20.csv"), R("shape_of_x_exaone_20.csv")]),
    pd.concat([R("scale_of_x.csv"), R("scale_of_x_exaone_20.csv")]))
core = syn[syn.model.isin(["GBM", "TabICLv2", "TabPFN-V3"])]
print("synthetic, %d runs = %d models x %d seeds"
      % (len(core), core.model.nunique(), core.seed.nunique()))

r = stats.paired_by_cluster(core, "scale_share", "shape_share", cluster="seed")
print("  rows      : %d of %d, Wilcoxon p = %.2g   <- pseudoreplicated"
      % (r["wins"], r["n"], r["naive_p"]))
print("  seeds     : %d of %d, sign test p = %.2g"
      % (r["cluster_wins"], r["clusters"], r["cluster_p"]))
print()
for m, g in core.groupby("model"):
    rm = stats.paired_by_cluster(g, "scale_share", "shape_share", cluster="seed")
    print("  %-10s %d of %d seeds, p = %.2g   (these are independent draws)"
          % (m, rm["cluster_wins"], rm["clusters"], rm["cluster_p"]))
print()
print("The per-model tests over 20 seeds are clean, because seeds are independent")
print("draws of the data. It is only the pooled test over 60 rows that counts the")
print("same 20 datasets three times.")


synthetic, 60 runs = 3 models x 20 seeds
  rows      : 59 of 60, Wilcoxon p = 2e-11   <- pseudoreplicated
  seeds     : 20 of 20, sign test p = 9.5e-07

  GBM        19 of 20 seeds, p = 2e-05   (these are independent draws)
  TabICLv2   20 of 20 seeds, p = 9.5e-07   (these are independent draws)
  TabPFN-V3  20 of 20 seeds, p = 9.5e-07   (these are independent draws)

The per-model tests over 20 seeds are clean, because seeds are independent
draws of the data. It is only the pooled test over 60 rows that counts the
same 20 datasets three times.


## 6c. Is the real-data reference stronger than the feature selection?

The feature carrying the shape gradient is chosen as the column with the largest
`|xi(t3) - xi(t1)|` on the fit split, a maximum over every column. The maximum of
many noisy statistics is large even when nothing is there, so the selection has a
null distribution of its own, measured by repeating it on permuted targets.

The same cell also compares the reference measured by Hill on residuals with the
reference measured by inverting the same quantile ratio the model is asked for,
which is the instrument the model side uses.


In [10]:
try:
    ri = tables.ok_rows(R("reference_instrument.csv"))
except Exception as e:
    print("not measured yet (run/run_reference_instrument.sh):", e)
    ri = None

if ri is not None and len(ri):
    print("reference gradient by instrument, median per dataset:")
    print(ri.groupby("dataset")[["ref_range_hill", "ref_range_ratio",
                                 "ref_range_ratio_residual", "null_p95"]]
          .median().round(3).to_string())
    print()
    share = ri.groupby("dataset").above_null.mean()
    print("share of runs whose gradient exceeds the 95th percentile of the")
    print("selection null, per dataset:")
    print(share.round(2).to_string())
    print()
    print("%d of %d datasets clear their own selection noise in the majority of runs."
          % (int((share > 0.5).sum()), len(share)))
    print("Datasets below 0.5 carry no rank claim beyond the selection itself, and")
    print("the text has to say so.")


reference gradient by instrument, median per dataset:
                               ref_range_hill  ref_range_ratio  ref_range_ratio_residual  null_p95
dataset                                                                                           
218_house_8L                           -0.297           -0.177                    -0.267     0.297
Allstate_Claims_Severity               -0.019           -0.059                    -0.056     0.057
Buzzinsocialmedia_Twitter              -0.253           -0.700                    -0.786     0.245
CPS1988                                -0.061           -0.073                    -0.013     0.061
OnlineNewsPopularity                   -0.011           -0.102                    -0.117     0.145
diamonds                               -0.008           -0.007                    -0.176     0.140
house_16H                              -0.178           -0.365                    -0.326     0.159
houses                                 -0.072          

## 6d. The same claim without a chosen feature and without a quotient

One fit per (dataset, model, seed), read through the terciles of every usable feature. The
share becomes the slope of the model's gradient on the reference gradient across features,
corrected for measurement error in the reference. That correction is the point: error in a
regressor drags a slope towards zero, and the shape reference is far noisier than the
scale one, so without it a worse instrument alone could produce the dissociation.


In [11]:
try:
    sl = tables.ok_rows(R("dissociation_real_slope.csv"))
except Exception as e:
    print("not measured yet (run/run_dissociation_real_slope.sh):", e)
    sl = None

if sl is not None and len(sl):
    print("%d runs, %d datasets, models %s"
          % (len(sl), sl.dataset.nunique(), sorted(sl.model.unique())))
    print()
    band = lambda c: sl[c].between(0.85, 1.15).sum()
    print("                                        scale     shape")
    print("  corrected slope inside 0.85 to 1.15   %2d/%-2d     %2d/%-2d"
          % (band("slope_scale_corrected"), sl.slope_scale_corrected.notna().sum(),
             band("slope_shape_corrected"), sl.slope_shape_corrected.notna().sum()))
    print("  median                                %.2f      %.2f"
          % (sl.slope_scale_corrected.median(), sl.slope_shape_corrected.median()))
    print("  range                                 %.2f to %.2f   %.1f to %.1f"
          % (sl.slope_scale_corrected.min(), sl.slope_scale_corrected.max(),
             sl.slope_shape_corrected.min(), sl.slope_shape_corrected.max()))
    print("  correlation with the reference        %.2f      %.2f"
          % (sl.r_scale.median(), sl.r_shape.median()))
    print("  measurement error, share of variance  %.3f     %.3f"
          % (sl.var_share_scale.median(), sl.var_share_shape.median()))
    print()
    r = stats.paired_by_cluster(sl, "slope_scale_corrected", "slope_shape_corrected",
                                cluster="dataset")
    print("scale closer to 1 than shape: %d of %d rows, %d of %d datasets, p = %.3g"
          % (r["wins"], r["n"], r["cluster_wins"], r["clusters"], r["cluster_p"]))
    print()
    print("The scale is tracked precisely and the shape is not tracked in any reliable")
    print("direction, which is a different statement from 'captures a smaller share'.")


27 runs, 9 datasets, models ['GBM']

                                        scale     shape
  corrected slope inside 0.85 to 1.15   24/27      1/23
  median                                0.95      1.86
  range                                 0.49 to 1.07   -5.8 to 6.6
  correlation with the reference        1.00      0.42
  measurement error, share of variance  0.002     0.180



scale closer to 1 than shape: 22 of 23 rows, 9 of 9 datasets, p = 0.00195

The scale is tracked precisely and the shape is not tracked in any reliable
direction, which is a different statement from 'captures a smaller share'.


## 7. Ensemble aggregation flattens the tail

Averaging quantiles across ensemble members is sharper than averaging probabilities, so
it compresses the tail. Dropping to one member is the one free recommendation in the
thesis, so its spread matters: it currently rests on few seeds.

In [12]:
v = tables.ok_rows(R("vincentization.csv"))
print(v.pivot_table(index="n_estimators", columns="model", values="xi_implied",
                    aggfunc="mean").round(3).to_string())
print()
g = v.groupby(["model", "n_estimators"]).seed.nunique()
print("seeds per cell: %d to %d" % (g.min(), g.max()))
print("The findings record three failures caused by a median over few seeds.")

model           GBM  TabICLv2  TabPFN-V3
n_estimators                            
1             0.606     0.718      0.546
2               NaN     0.708      0.449
4               NaN     0.711      0.447
8               NaN     0.714      0.446

seeds per cell: 1 to 3
The findings record three failures caused by a median over few seeds.


## 8. The figure

In [13]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.2, 3.6))
order = list(tab.dropna(subset=["shape n_est=1"]).sort_values("scale").index)
y = np.arange(len(order))
ax.hlines(y, tab.loc[order, "shape n_est=1"], tab.loc[order, "scale"],
          color="0.75", lw=3, zorder=1)
ax.scatter(tab.loc[order, "shape n_est=1"], y, s=55, label="tail shape", zorder=2)
ax.scatter(tab.loc[order, "scale"], y, s=55, marker="D", label="scale", zorder=2)
ax.axvline(1.0, ls=":", c="0.4", lw=1)
ax.set_yticks(y); ax.set_yticklabels(order)
ax.set_xlabel("share of the true conditional change the model reproduces")
ax.set_xlim(0, 1.15)
ax.legend(loc="lower right", frameon=False)
ax.set_title("Conditional scale is tracked, conditional tail shape is not")
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(paths.figure("dissociation." + ext), dpi=200)
print("wrote figures/dissociation.png and .pdf")
fig

wrote figures/dissociation.png and .pdf


<Figure size 720x360 with 1 Axes>

## Disagreements with the text

Anything the recomputation could not confirm is listed here rather than left silent.

In [14]:
notes = [
 "shape share at one ensemble member is 42 to 45% for the three core models and "
 "29 to 45% across all five; the application should quote the wider range.",
 "the 9.1% pinball gain from a single ensemble member rests on 3 seeds, one model "
 "and one xi; run/ has a driver to raise it to 10 seeds before it is quoted.",
 "sample_complexity.csv was measured with the test set drawn after the training set "
 "from one stream, so its scatter across n_train is partly an artefact; the script "
 "now uses a separate stream and the table above is from the old file.",
 "the p-values that were quoted (3.4e-11 real, 2.2e-11 synthetic) are row-level and "
 "pseudoreplicated; the cluster-level equivalents are p = 0.002 over 9 datasets and "
 "p = 9.5e-07 over 20 seeds, and those are the ones the text now uses.",
 "on real data the reference gradient and the model's are produced by DIFFERENT "
 "estimators, Hill on residuals against a ratio inversion; reference_instrument.csv "
 "measures both on the same terciles so the mismatch can be quoted rather than assumed.",
 "shape_share is missing in 28 of 90 real-data rows and scale_share in 6, because the "
 "guard |ref_range| < 0.05 refuses a reference gradient too small to divide by. The "
 "rows drop per (dataset, seed) and identically for all three models, so the model "
 "comparison is unaffected, but the surviving subset is the one with the strongest "
 "shape gradient. Allstate_Claims_Severity contributes 0 of its 9 rows.",
]
for n in notes:
    print("-", n)


- shape share at one ensemble member is 42 to 45% for the three core models and 29 to 45% across all five; the application should quote the wider range.
- the 9.1% pinball gain from a single ensemble member rests on 3 seeds, one model and one xi; run/ has a driver to raise it to 10 seeds before it is quoted.
- sample_complexity.csv was measured with the test set drawn after the training set from one stream, so its scatter across n_train is partly an artefact; the script now uses a separate stream and the table above is from the old file.
- the p-values that were quoted (3.4e-11 real, 2.2e-11 synthetic) are row-level and pseudoreplicated; the cluster-level equivalents are p = 0.002 over 9 datasets and p = 9.5e-07 over 20 seeds, and those are the ones the text now uses.
- on real data the reference gradient and the model's are produced by DIFFERENT estimators, Hill on residuals against a ratio inversion; reference_instrument.csv measures both on the same terciles so the mismatch can be q